In [11]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [12]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle API key configured.")

Kaggle API key configured.


In [13]:
!pip install kaggle wfdb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [14]:
!kaggle datasets download -d khyeh0719/ptb-xl-dataset -p /content/ptbxl --unzip

Dataset URL: https://www.kaggle.com/datasets/khyeh0719/ptb-xl-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.72G/1.72G [00:16<00:00, 111MB/s]



In [20]:
import os

base_path = "/content/ptbxl"

dataset_folder = os.listdir(base_path)[0]

ptbxl_path = os.path.join(base_path, dataset_folder)

print("Dataset Path:")
print(ptbxl_path)

Dataset Path:
/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1


In [21]:
import pandas as pd
import numpy as np
import ast
import os

# Automatically locate the dataset folder
base_path = "/content/ptbxl"
dataset_folder = os.listdir(base_path)[0]
ptbxl_path = os.path.join(base_path, dataset_folder)

print("Dataset Path:")
print(ptbxl_path)

# Load PTB-XL metadata
df = pd.read_csv(
    os.path.join(ptbxl_path, "ptbxl_database.csv"),
    index_col="ecg_id"
)

# Convert string dictionaries into Python dictionaries
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

print("\nDataset Shape:", df.shape)

df.head()

Dataset Path:
/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1

Dataset Shape: (21837, 27)


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [22]:
print("\nColumns:")
print(df.columns.tolist())

print("\nNumber of ECG Records:", len(df))

print("\nUnique Patients:", df.patient_id.nunique())

print("\nSampling Frequencies Available:")
print(df["filename_lr"].notna().sum(), "Low Resolution (100Hz)")
print(df["filename_hr"].notna().sum(), "High Resolution (500Hz)")


Columns:
['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

Number of ECG Records: 21837

Unique Patients: 18885

Sampling Frequencies Available:
21837 Low Resolution (100Hz)
21837 High Resolution (500Hz)


In [23]:
# ==========================================
# Missing Values in Metadata
# ==========================================

missing_df = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().mean() * 100).round(2)
})

missing_df = missing_df[
    missing_df["Missing Values"] > 0
].sort_values(
    by="Percentage",
    ascending=False
)

missing_df

,Missing Values,Percentage
electrodes_problems,21807,99.86
infarction_stadium2,21734,99.53
pacemaker,21544,98.66
burst_noise,21224,97.19
baseline_drift,20230,92.64
extra_beats,19883,91.05
static_noise,18575,85.06
infarction_stadium1,16211,74.24
height,14854,68.02
weight,12408,56.82


In [24]:
agg_df = pd.read_csv(
    os.path.join(ptbxl_path, "scp_statements.csv"),
    index_col=0
)

agg_df = agg_df[
    agg_df["diagnostic"] == 1
]

def aggregate_diagnostic(scp_codes_dict):

    superclasses = []

    for code in scp_codes_dict.keys():

        if code in agg_df.index:

            superclasses.append(
                agg_df.loc[
                    code,
                    "diagnostic_class"
                ]
            )

    return sorted(
        list(
            set(superclasses)
        )
    )

df["diagnostic_superclass"] = df["scp_codes"].apply(
    aggregate_diagnostic
)

print(df["diagnostic_superclass"].head(10))

ecg_id
1     [NORM]
2     [NORM]
3     [NORM]
4     [NORM]
5     [NORM]
6     [NORM]
7     [NORM]
8       [MI]
9     [NORM]
10    [NORM]
Name: diagnostic_superclass, dtype: object


In [25]:
def map_to_binary(superclass_list):
    if len(superclass_list) == 0:
        return np.nan  # no diagnostic info at all — drop these later
    if set(superclass_list) == {"NORM"}:
        return 0
    return 1  # anything containing MI, STTC, CD, or HYP (even mixed with NORM) = disease

df["target_binary"] = df["diagnostic_superclass"].apply(map_to_binary)

print("Before removing unlabeled ECGs")
print(df["target_binary"].value_counts(dropna=False))

print()

df = df.dropna(subset=["target_binary"])

df["target_binary"] = df["target_binary"].astype(int)

print("After removing unlabeled ECGs")
print(df["target_binary"].value_counts())

print()

print("Remaining ECGs:", len(df))
print(df.shape)

Before removing unlabeled ECGs
target_binary
1.0    12347
0.0     9083
NaN      407
Name: count, dtype: int64

After removing unlabeled ECGs
target_binary
1    12347
0     9083
Name: count, dtype: int64

Remaining ECGs: 21430
(21430, 29)


In [28]:
from sklearn.model_selection import train_test_split

df_subset, _ = train_test_split(
    df, train_size=5000, stratify=df["target_binary"], random_state=42
)
print("="*60)
print("Diagnostic Superclass Distribution")
print("="*60)

print(df["diagnostic_superclass"].explode().value_counts())

print("Subset shape:", df_subset.shape)
print(df_subset["target_binary"].value_counts(normalize=True))

Diagnostic Superclass Distribution
diagnostic_superclass
NORM    9528
MI      5486
STTC    5250
CD      4907
HYP     2655
Name: count, dtype: int64
Subset shape: (5000, 29)
target_binary
1    0.5762
0    0.4238
Name: proportion, dtype: float64


In [29]:
print("Unique Patients in Subset :", df_subset["patient_id"].nunique())
print("Total ECGs in Subset      :", len(df_subset))

Unique Patients in Subset : 4807
Total ECGs in Subset      : 5000


In [30]:
df_subset_save = df_subset[
    [
        "patient_id",
        "filename_lr",
        "filename_hr",
        "diagnostic_superclass",
        "target_binary"
    ]
].copy()

df_subset_save.to_csv(
    "06_ecg_subset_metadata.csv",
    index=True
)

print(df_subset_save.shape)

df_subset_save.head()

(5000, 5)


,patient_id,filename_lr,filename_hr,diagnostic_superclass,target_binary
ecg_id,,,,,
9625,18700.0,records100/09000/09625_lr,records500/09000/09625_hr,[NORM],0
15843,8908.0,records100/15000/15843_lr,records500/15000/15843_hr,[MI],1
2961,2236.0,records100/02000/02961_lr,records500/02000/02961_hr,[NORM],0
13763,9227.0,records100/13000/13763_lr,records500/13000/13763_hr,[MI],1
3725,11961.0,records100/03000/03725_lr,records500/03000/03725_hr,[CD],1


In [31]:
from google.colab import files

files.download("06_ecg_subset_metadata.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>